# 第12次课：异常与调试（模块二收官）

**地学编程基础** | 2学时（90分钟）

---

## 🎓 模块二即将完结

回顾模块二走过的 4 节课：

| 次数 | 主题 | 核心成果 |
|---|---|---|
| 第 9 次 | 函数基础 | 把代码打包成可重用的工具 |
| 第 10 次 | 函数进阶 | lambda、import、docstring |
| 第 11 次 | 文件读写 | 对接真实数据文件 |
| 第 12 次（今天）| 异常与调试 | 让代码更健壮、更专业 |

**今天结束后**——模块二全部完成——你的代码能力就到达**工程级别**了！

---

## 📚 本节课学习目标

完成本节课后，你应该能够：

1. **理解**异常的本质（程序遇到意外情况）；
2. **认识** 6 种最常见的异常类型；
3. **掌握** `try-except` 处理异常的基础语法；
4. **掌握**多种异常分别捕获的写法；
5. **理解** `try-except-else-finally` 完整结构；
6. **能够**读懂 Python 的 Traceback 错误信息；
7. **掌握** `print` 调试和 Jupyter 单元格调试技巧；
8. **能够**给文件读写代码加上完整的异常处理。

---

## 🤔 你早就遇到过异常

**前几次课，你已经遇到过很多'报错'**：

- 第7次课：访问字典里不存在的键 → `KeyError`
- 第11次课：打开不存在的文件 → `FileNotFoundError`
- 第11次课：用错编码读取中文文件 → `UnicodeDecodeError`
- 用 int 转换非数字字符串 → `ValueError`
- 列表索引越界 → `IndexError`
- 除以 0 → `ZeroDivisionError`

**之前我们的策略**：用 if-in 判断、用 get()、避免触发错误。

**今天的策略**：用 `try-except` **优雅处理'已经发生的错误'**——让程序能从错误中恢复，而不是崩溃。

---

## 一、什么是异常？

### 1.1 异常的本质

**异常（Exception）**：程序运行时遇到的'意外情况'。

**例子**：
- 程序想读一个文件——但文件不存在
- 程序想做除法——但除数是 0
- 程序想把字符串转成数字——但字符串是 "hello"

**当 Python 遇到异常时**：
1. 程序**立即停止**（除非你处理它）
2. 显示一段叫 **Traceback** 的错误信息
3. 退出当前函数/单元格的执行

**今天的目标**：让程序'**有能力面对异常，而不是崩溃**'。

### 1.2 常见的 6 种异常类型

**让我们一次性见识它们**：

In [ ]:
# 1. ValueError —— 值的类型对，但值不合法
# int("hello")    # ValueError: invalid literal for int() with base 10: 'hello'

# 2. TypeError —— 类型错误
# "a" + 1         # TypeError: can only concatenate str (not "int") to str

# 3. KeyError —— 字典中没有这个键
d = {"a": 1}
# print(d["b"])   # KeyError: 'b'

# 4. IndexError —— 列表索引越界
lst = [1, 2, 3]
# print(lst[100]) # IndexError: list index out of range

# 5. ZeroDivisionError —— 除以 0
# 10 / 0          # ZeroDivisionError: division by zero

# 6. FileNotFoundError —— 文件不存在
# open("data/不存在的文件.txt")    # FileNotFoundError: ...

# （上面用 # 注释掉的代码，去掉 # 运行就能看到对应异常）
print("上面 6 个例子都是会触发异常的代码。")
print("请把任意一行的 # 去掉，运行试试看 Traceback。")

**📌 这 6 种异常是地学/数据处理代码里最常见的**——

| 异常 | 触发场景 | 防御策略 |
|---|---|---|
| `ValueError` | int("abc")、float("") | 检查输入格式 |
| `TypeError` | 字符串+数字混用 | 检查类型 |
| `KeyError` | d["不存在的键"] | get() 或 in |
| `IndexError` | lst[100]（列表才 5 个元素）| 检查长度 |
| `ZeroDivisionError` | 10 / 0 | 检查除数 |
| `FileNotFoundError` | 文件不存在 | 用 try-except |

---

## 二、try-except 基础

### 2.1 第一个 try-except 例子

**🔑 语法**：

```python
try:
    可能出错的代码
except 异常类型:
    出错时执行的代码
```

**意思**：'**试着运行 try 块里的代码——如果出错了，跳到 except 块**'。

In [ ]:
# 没有异常处理 ——
# 如果用户输入'abc'，整个程序崩溃

# user_input = input("请输入一个数字：")
# number = int(user_input)        # 输入 'abc' → ValueError
# print(f"你输入的数字是 {number}")

# 用 try-except ——
# 即使输入错误，程序也能优雅恢复
user_input = "abc"    # 模拟用户错误输入

try:
    number = int(user_input)
    print(f"你输入的数字是 {number}")
except ValueError:
    print("输入错误：必须是数字！")

print("程序继续执行...")    # 程序不会崩溃

**关键观察**：
- `try` 块里的代码出错了——程序**不崩溃**
- 跳到 `except` 块执行'兜底'代码
- **try 块中错误之后的代码不会执行**（`print(f"你输入的数字...")` 没运行）
- 整个 try-except 之后的代码继续执行（`print("程序继续执行...")`）

### 2.2 try 块里的代码可以多行

`try` 块里可以写多行代码——**任何一行出错都会跳到 except**：

In [ ]:
raw = "温度: "    # 故意提供格式不对的数据

try:
    # 这一段代码可能在多个地方出错
    parts = raw.split(":")
    label = parts[0].strip()
    value = float(parts[1])    # parts[1] 是空字符串 → ValueError
    print(f"{label}: {value}")
except ValueError:
    print("数据格式错误，跳过这条记录")

### 2.3 获取异常的具体信息：as e

In [ ]:
# 用 as e 拿到异常对象 —— 可以打印更多细节
try:
    number = int("hello")
except ValueError as e:
    print(f"出错了！具体原因：{e}")

**`as e`**：把异常对象赋值给 `e`——可以 `print(e)` 看具体信息。

**很有用**——你能告诉用户'**到底哪里错了**'，而不是只说'出错了'。

### 2.4 ⚠️ 不推荐：裸 except（捕获所有异常）

In [ ]:
# ❌ 不推荐：捕获所有异常（包括程序员的 bug）
try:
    x = int("hello")
except:    # 捕获所有 ——包括语法错、键盘中断等
    print("出错了")    # 但你不知道具体是什么错

In [ ]:
# ✅ 推荐：明确指定要处理的异常类型
try:
    x = int("hello")
except ValueError as e:
    print(f"值错误：{e}")

**🛡️ 防御口诀**：
- **永远明确指定要捕获的异常类型** —— 不要用裸 except
- 这样**只捕获预期的错误**，**程序员的 bug 该崩溃就崩溃**（容易调试）

---

## 🎯 即时练习①

**练习目标**：熟悉 try-except 基础。

**练习题**：写一个'安全的字符串转数字'函数 `safe_int(s, default=0)`：

- 输入字符串 s 和默认值（默认 0）
- 如果能转成 int，返回 int 值
- 如果不能转（ValueError），返回默认值
- 函数永远不会抛出异常

**测试**：
```python
print(safe_int("42"))         # 42
print(safe_int("hello"))      # 0（默认）
print(safe_int("hello", -1))  # -1
print(safe_int("3.14"))       # 0（int 不能直接转 "3.14"）
```

In [ ]:
# ===== 即时练习① =====



# 测试
# print(safe_int("42"))
# print(safe_int("hello"))
# print(safe_int("hello", -1))
# print(safe_int("3.14"))

---

## 三、多种异常分别捕获

**真实代码可能触发多种不同的异常**——可以**分别处理**：

**🔑 语法**：多个 `except` 块

In [ ]:
def safe_divide(s1, s2):
    """把两个字符串转成数字相除，处理可能的多种异常。"""
    try:
        a = float(s1)
        b = float(s2)
        result = a / b
        return result
    except ValueError:
        print("输入不是有效数字")
        return None
    except ZeroDivisionError:
        print("除数不能为 0")
        return None

# 测试
print(safe_divide("10", "2"))      # 5.0
print(safe_divide("hello", "2"))   # 输入不是有效数字
print(safe_divide("10", "0"))      # 除数不能为 0

**关键观察**：
- 每个 `except` 处理一种异常
- Python 会**按顺序匹配**——遇到第一个匹配的 except 块就执行，**其他 except 块不会再执行**
- 如果**没有 except 匹配**——异常会继续向上抛出

### 3.1 多个异常用一个 except 处理（用元组）

In [ ]:
# 如果两种异常处理方式相同，可以合并
def safe_divide_v2(s1, s2):
    try:
        return float(s1) / float(s2)
    except (ValueError, ZeroDivisionError) as e:    # ← 用元组合并
        print(f"出错了：{e}")
        return None

print(safe_divide_v2("hello", "2"))
print(safe_divide_v2("10", "0"))

**📌 经验法则**：
- **不同异常处理方式不同** → 多个 except 块
- **不同异常处理方式相同** → 一个 except + 元组

---

## 四、try-except-else-finally 完整结构

**Python 的 try 结构有 4 个块**——前 2 个我们已经学过，后 2 个是补充：

```python
try:
    # 可能出错的代码
except 异常:
    # 出错时执行
else:
    # try 块成功时执行（没出错）
finally:
    # 无论如何都会执行
```

### 4.1 else —— 没出错时执行

In [ ]:
# else：try 块成功时（没异常）执行
user_input = "42"

try:
    number = int(user_input)
except ValueError:
    print("输入错误")
else:
    # 只有 try 成功才会执行
    print(f"成功转换：{number}")
    print("可以放心使用 number 了")

**为什么要有 else？**

把'**成功后才能做**'的代码放在 else 里——**让 try 块更小**——只放真正可能出错的语句。

**注意**：else 是**可选**的——不写也没问题。

### 4.2 finally —— 无论如何都执行

**最有用的场景**：清理资源（关闭文件、断开数据库连接等）。

In [ ]:
# finally：无论是否出错都执行（适合做'清理'工作）
try:
    print("打开资源...")
    x = int("hello")    # 这一行会出错
    print("使用资源...")
except ValueError:
    print("出错了")
finally:
    print("关闭资源")    # 一定会执行！

**📌 现在你知道了**：第11次课的 `with open()` —— **背后用的就是 finally 机制**——保证文件一定会关闭，**即使 with 块里的代码出错**。

### 4.3 完整流程图

```
      ┌─────────┐
      │   try    │
      └────┬────┘
           │
      ┌────┴────┐
      │  出错？  │
      └─┬─────┬─┘
      Yes      No
        ↓       ↓
    ┌─────┐  ┌─────┐
    │except│  │ else│
    └──┬──┘  └──┬──┘
       └──┬──┬──┘
          ↓  ↓
       ┌────────┐
       │ finally │   （永远执行）
       └────────┘
```

---

## 🎯 即时练习②

**练习目标**：综合运用 try-except 处理文件读取错误。

**练习题**：写一个函数 `read_elevations(filepath)`：

- 输入文件路径
- 读取文件，解析每行成 `(name, elevation)` 元组
- 返回元组列表
- **要求**：
  - 文件不存在时（`FileNotFoundError`）—— 打印友好提示，返回空列表 `[]`
  - 某一行格式错误（不能 int 转换）—— 打印警告，**跳过这一行**继续处理其他行

**测试**：
```python
data = read_elevations("data/elevations.txt")    # 正常 —— 返回完整列表
data = read_elevations("不存在的文件.txt")       # 友好提示 —— 返回 []
```

**提示**：
- 用 `try-except FileNotFoundError` 包住整个文件操作
- 用一个**内部的** `try-except ValueError` 处理每行的解析

In [ ]:
# ===== 即时练习② =====





---

## 五、读懂 Traceback 错误信息

**Python 报错时的那一大段红字——叫 Traceback**——很多新手看到就害怕。

**今天教你读懂它**——**Traceback 是你最好的调试朋友**。

### 5.1 一个典型的 Traceback

In [ ]:
def process_data(data):
    return calculate(data["values"])

def calculate(values):
    return sum(values) / len(values)

# 故意触发错误
data = {"name": "测试"}     # 没有 "values" 键
# process_data(data)         # 取消这行的 # 看 Traceback

# 看 Traceback 大概长这样：
# Traceback (most recent call last):
#   File "...", line 9, in <module>
#     process_data(data)
#   File "...", line 2, in process_data
#     return calculate(data["values"])
# KeyError: 'values'

print("请把上面的注释 # 去掉，看真实的 Traceback")

### 5.2 Traceback 的'解读步骤'

**🔑 三步法**：

**第 1 步：看最后一行——异常类型 + 描述**
```
KeyError: 'values'
```
→ '字典里没有 values 这个键'

**第 2 步：从下往上看——找'你的代码'里出错的行**
```
File "...", line 2, in process_data
    return calculate(data["values"])    ← 在这一行！
```
→ 第 2 行 `data["values"]` 找不到键

**第 3 步：再往上看——理解调用链**
```
File "...", line 9, in <module>
    process_data(data)                  ← 是这里调用的
```
→ 是从第 9 行的 `process_data(data)` 调用过来的

**📌 经验法则**：**总是先看 Traceback 的最后一行**——异常类型 + 描述——这是定位问题最快的方式。

---

## 六、调试技巧

**调试 = 找出代码为什么行为和预期不一致**。

今天介绍 **3 个最实用的调试方法**。

### 6.1 print 调试 —— 最简单也最有效

**99% 的调试场景，print 就够用**。

**核心思路**：在你怀疑出问题的地方加 `print` 看变量的值。

In [ ]:
# 假设你的代码结果不对，怎么调试？

def calculate_average_temp(temps):
    """计算平均温度"""
    print(f"DEBUG: 输入参数 temps = {temps}")    # ← 看输入对不对
    
    total = 0
    for t in temps:
        print(f"DEBUG: 累加 {t}")                # ← 看每步的过程
        total += t
    
    print(f"DEBUG: 总和 = {total}")              # ← 看中间结果
    avg = total / len(temps)
    print(f"DEBUG: 平均 = {avg}")                # ← 看最终结果
    return avg

result = calculate_average_temp([28, 32, 35, 30])
print(f"最终结果：{result}")

**🛡️ print 调试的最佳实践**：

1. **在关键步骤打印变量值** —— 看每步是否符合预期
2. **用 `f"DEBUG: ..."` 前缀** —— 调试信息和正常输出区分开
3. **打印类型** —— `print(type(x), x)` 帮助定位类型错误
4. **调试完记得删掉/注释** —— 不要污染最终代码

### 6.2 Jupyter 单元格调试 —— 拆分代码逐步看

**Jupyter Notebook 的核心优势**——**能把代码拆成多个单元格，逐个执行、逐个看结果**。

In [ ]:
# 比如调试一段复杂的代码 ——
# 把它拆成多个单元格，每个单元格输出中间结果

# 单元格1：读取数据
raw = "温度: 32.5°C"
print(f"原始数据：{raw}")

In [ ]:
# 单元格2：分隔字段
parts = raw.split(":")
print(f"分隔后：{parts}")
print(f"长度：{len(parts)}")

In [ ]:
# 单元格3：取值并清理
value_str = parts[1].strip()
print(f"清理后：{repr(value_str)}")    # repr 显示真实字符串

In [ ]:
# 单元格4：去掉单位
value_str = value_str.replace("°C", "")
print(f"去单位后：{repr(value_str)}")

In [ ]:
# 单元格5：转换为浮点数
value = float(value_str)
print(f"最终值：{value}")

**🌟 这就是 Jupyter 调试的神技**——
- **每个单元格只做一件事**
- **每步都能看到结果**
- **出错了能精确定位到哪个单元格**
- **可以反复修改某个单元格再运行**

### 6.3 用 type() 查看类型

**新手最常见的隐藏 bug**——**类型错误**。

In [ ]:
# 经典 bug：CSV 读出来的数字其实是字符串！
row = {"city": "北京", "elevation": "43.5"}    # 模拟 CSV 读出来的

# 想计算海拔的两倍
result = row["elevation"] * 2
print(f"结果：{result}")    # '43.543.5' —— 字符串重复，不是数学运算！
print(f"类型：{type(row['elevation'])}")    # str —— 原来是字符串！

# 修正：先转成数字
result = float(row["elevation"]) * 2
print(f"修正后：{result}")

**🛡️ 调试时多用 `type()`**——尤其当结果不符合预期但又不报错的时候。

**还有一些其他工具**（了解即可）：
- `len(x)` —— 看长度
- `repr(x)` —— 看原始表示（包括 `\n` 等）
- `print(x.__dict__)` —— 看对象的所有属性

---

## 七、raise —— 主动抛出异常（点到为止）

**有时候你的函数发现输入有问题——主动抛出异常，让调用者知道**。

In [ ]:
def calculate_bmi(weight, height):
    """计算 BMI 指数。要求 weight > 0，height > 0。"""
    
    # 参数验证：发现问题主动抛异常
    if weight <= 0:
        raise ValueError(f"体重必须为正，但传入了 {weight}")
    if height <= 0:
        raise ValueError(f"身高必须为正，但传入了 {height}")
    
    return weight / (height ** 2)

# 正常调用
print(calculate_bmi(70, 1.75))    # 22.857...

# 错误调用 —— 异常会被'抛出'
try:
    print(calculate_bmi(-50, 1.75))
except ValueError as e:
    print(f"参数错误：{e}")

**📌 何时用 raise？**
- **参数验证失败** —— 立即报告错误（'**早失败**'原则）
- **检测到不该出现的状态** —— 让程序停下来，不要继续错下去

**🛡️ 经验法则**：**写函数时，对'非法输入'尽早 raise**——比让函数返回错误结果好得多。

---

## 🎯 即时练习③（综合实战）

**练习目标**：把第11次课的代码加上完整的异常处理。

**情境**：第11次课你写过这段代码——**读取观测站文件**：

```python
stations = []
with open("data/elevations.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        name = parts[0]
        elevation = int(parts[1])
        stations.append((name, elevation))
```

**今天的任务**：把它升级为'**健壮版本**'——封装成函数 `load_stations(filepath)`：

1. **文件不存在**（`FileNotFoundError`）—— 打印友好提示，返回 `[]`；
2. **某行解析错误**（`ValueError`/`IndexError`）—— 打印警告（含行号 + 内容），**跳过该行**继续处理；
3. **统计**：解析成功多少行、跳过多少行；
4. **测试**：
   - 用 `data/elevations.txt`（正常文件）
   - 用 `不存在的文件.txt`（看友好提示）
   - 用一个**故意有坏数据**的文件 `data/messy.txt`（看跳过功能）

**步骤提示**：
- 整体用 try-except 包住 open
- 内部 for 循环里再用 try-except 包住每行解析
- 用 `enumerate` 拿到行号方便报告

In [ ]:
# ===== 即时练习③ 主函数 =====

def load_stations(filepath):
    """健壮地加载观测站数据。
    
    返回 (stations 列表, 跳过的行数)
    """
    pass    # 由你来实现


# 测试1：正常文件


# 测试2：不存在的文件


# 测试3：先生成一个故意有坏数据的文件 messy.txt
messy_content = """塔克拉玛干 1200
敦煌 abc
格尔木 2800
拉萨
林芝 2900
广州 50
"""

with open("data/messy.txt", "w", encoding="utf-8") as f:
    f.write(messy_content)

# 然后调用 load_stations 测试




---

## 八、本节课小结

### 知识点回顾

| 知识点 | 关键语法 |
|---|---|
| 6 种常见异常 | ValueError、TypeError、KeyError、IndexError、ZeroDivisionError、FileNotFoundError |
| try-except 基础 | `try: ... except 异常: ...` |
| 获取异常详情 | `except ValueError as e:` |
| 多种异常分别处理 | 多个 except 块 |
| 多种异常合并处理 | `except (A, B):` 用元组 |
| 完整结构 | `try-except-else-finally` |
| 主动抛异常 | `raise ValueError(...)` |
| 调试 | print + Jupyter 单元格 + type |
| 读 Traceback | 最后一行看异常类型 + 从下往上看代码 |

### 重点强调（重要程度从高到低）

1. ⭐⭐⭐ **`try-except 异常类型`** —— 永远明确指定异常类型
2. ⭐⭐⭐ **读 Traceback 先看最后一行** —— 异常类型 + 描述定位最快
3. ⭐⭐⭐ **print 调试 + 单元格调试** —— 99% 的 bug 都能这样找出来
4. ⭐⭐ **for 循环里加内层 try-except** —— 跳过坏数据的标准模式
5. ⭐⭐ **finally 用于清理资源** —— 文件、连接等
6. ⭐ **裸 except 不推荐** —— 会捕获程序员的 bug
7. ⭐ **raise 主动抛异常** —— 函数验证参数的标准做法

### 课后作业

**1. 【基础】安全的数据转换函数（必做）**

写以下 3 个函数，每个都用 try-except：

1. `safe_float(s, default=0.0)`：字符串转 float，失败返回默认值
2. `safe_get(d, key, default=None)`：字典安全访问（其实就是 dict.get，但你自己用 try-except 实现）
3. `safe_divide(a, b)`：a / b，除以 0 返回 None

**2. 【进阶】健壮的 CSV 读取（必做）**

把第11次课的 `data/cities_climate.csv` 读取代码升级：

1. 文件不存在时友好提示
2. 某行字段缺失时跳过
3. 某行字段类型错误（如纬度不是数字）时跳过
4. 输出统计：成功多少行、跳过多少行、总行数

**3. 【挑战】参数验证函数（必做）**

写一个 `geographic_distance` 函数（呼应第10次课）：

- 输入 (lat1, lon1, lat2, lon2)
- 用 raise ValueError 验证：
  - 纬度必须在 -90 到 90 之间
  - 经度必须在 -180 到 180 之间
- 验证通过后用第10次课的简化公式计算距离
- 测试 5 个不同的输入（包括非法的）

---

下节课预告：**第13次课 NumPy 基础** —— 进入**模块三（科学计算）**！我们会学习 NumPy 数组——比 Python 列表强大百倍的数据处理工具。

---
---

## 🎓 模块二完结撒花！

**模块二（函数+文件+异常）4 节课走过的路**：

| 课次 | 主题 | 你掌握的能力 |
|---|---|---|
| 9 | 函数基础 | 把代码打包成可重用工具 |
| 10 | 函数进阶 | lambda、import、docstring |
| 11 | 文件读写 | 对接真实数据文件 |
| 12 | 异常与调试 | 让代码健壮、能调试 |

**至此，你的代码能力已达到'工程级别'**：

✓ 写函数封装通用逻辑

✓ 用 lambda 让代码简洁优雅

✓ 读写真实的 CSV、TXT 文件

✓ 优雅处理异常，让程序'不崩溃'

✓ 用 print/单元格/Traceback 调试问题

**接下来——模块三（科学计算）**：

- 第 13 次：NumPy 基础（数组、向量化运算）
- 第 14 次：Pandas 基础（DataFrame）
- 第 15 次：Pandas 分组聚合

**模块三是 GIS/地学专业的核心工具入门**——12 次课打下的基础在这里全部派上用场！

---
---

# 📎 附录：参考答案

> **建议先独立完成所有练习题再查看答案。**

In [ ]:
# ===== 即时练习① 参考答案 =====
def safe_int(s, default=0):
    """安全的字符串转 int。
    
    参数：
        s (str): 要转换的字符串
        default: 转换失败时的默认值
    
    返回：
        int 或 default
    """
    try:
        return int(s)
    except ValueError:
        return default

# 测试
print(safe_int("42"))         # 42
print(safe_int("hello"))      # 0
print(safe_int("hello", -1))  # -1
print(safe_int("3.14"))       # 0（int 不能直接转 "3.14"）

In [ ]:
# ===== 即时练习② 参考答案 =====
def read_elevations(filepath):
    """健壮地读取观测站文件。
    
    返回：
        list: [(name, elevation), ...]，文件不存在时返回 []
    """
    stations = []
    
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                
                # 内层 try-except：跳过坏数据
                try:
                    parts = line.split()
                    name = parts[0]
                    elevation = int(parts[1])
                    stations.append((name, elevation))
                except (ValueError, IndexError) as e:
                    print(f"跳过坏数据：{line}（原因：{e}）")
    
    except FileNotFoundError:
        print(f"文件不存在：{filepath}")
    
    return stations

# 测试
print("=== 测试 1：正常文件 ===")
data = read_elevations("data/elevations.txt")
print(f"读取了 {len(data)} 条记录")
for d in data[:3]:
    print(d)

print("\n=== 测试 2：不存在的文件 ===")
data = read_elevations("不存在.txt")
print(f"返回：{data}")

In [ ]:
# ===== 即时练习③ 参考答案 =====
def load_stations(filepath):
    """健壮地加载观测站数据。
    
    返回：(stations 列表, 跳过的行数)
    """
    stations = []
    skipped = 0
    
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            for line_num, line in enumerate(f, 1):    # enumerate 拿行号
                line = line.strip()
                if not line:
                    continue
                
                try:
                    parts = line.split()
                    name = parts[0]
                    elevation = int(parts[1])
                    stations.append((name, elevation))
                except (ValueError, IndexError) as e:
                    print(f"⚠️  第 {line_num} 行跳过：{repr(line)}（{e}）")
                    skipped += 1
    
    except FileNotFoundError:
        print(f"❌ 文件不存在：{filepath}")
        return [], 0
    
    return stations, skipped

# === 测试1：正常文件 ===
print("=== 测试 1：正常文件 ===")
data, skipped = load_stations("data/elevations.txt")
print(f"成功读取：{len(data)} 行，跳过：{skipped} 行")
for d in data[:3]:
    print(" ", d)

# === 测试2：不存在的文件 ===
print("\n=== 测试 2：不存在的文件 ===")
data, skipped = load_stations("不存在.txt")
print(f"成功读取：{len(data)} 行")

In [ ]:
# === 测试3：先生成坏数据文件 messy.txt ===
import os
os.makedirs("data", exist_ok=True)

messy_content = """塔克拉玛干 1200
敦煌 abc
格尔木 2800
拉萨
林芝 2900
广州 50
"""

with open("data/messy.txt", "w", encoding="utf-8") as f:
    f.write(messy_content)

print("=== 测试 3：含坏数据的文件 ===")
data, skipped = load_stations("data/messy.txt")
print(f"\n📊 统计：成功 {len(data)} 行，跳过 {skipped} 行")
print("成功的数据：")
for d in data:
    print(" ", d)

---

## 课后作业参考答案

In [ ]:
# ===== 课后作业 1 参考答案 =====

def safe_float(s, default=0.0):
    """安全的字符串转 float。"""
    try:
        return float(s)
    except (ValueError, TypeError):
        return default

def safe_get(d, key, default=None):
    """用 try-except 实现的字典安全访问。"""
    try:
        return d[key]
    except KeyError:
        return default

def safe_divide(a, b):
    """安全除法，除以 0 返回 None。"""
    try:
        return a / b
    except ZeroDivisionError:
        return None

# 测试
print(safe_float("3.14"))        # 3.14
print(safe_float("abc"))         # 0.0
print(safe_float("abc", -1.0))   # -1.0

d = {"a": 1, "b": 2}
print(safe_get(d, "a"))          # 1
print(safe_get(d, "x"))          # None
print(safe_get(d, "x", 99))      # 99

print(safe_divide(10, 2))        # 5.0
print(safe_divide(10, 0))        # None

In [ ]:
# ===== 课后作业 2 参考答案 =====
import csv

def load_cities_robust(filepath):
    """健壮的 CSV 读取。"""
    cities = []
    success = 0
    skipped = 0
    
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for line_num, row in enumerate(reader, 2):    # 2 因为表头是第 1 行
                try:
                    cities.append({
                        "city": row["city"],
                        "lat": float(row["latitude"]),
                        "lon": float(row["longitude"]),
                        "temp": float(row["annual_temp"]),
                        "rain": float(row["annual_rainfall"]),
                    })
                    success += 1
                except (ValueError, KeyError) as e:
                    print(f"⚠️  第 {line_num} 行跳过：{e}")
                    skipped += 1
    
    except FileNotFoundError:
        print(f"❌ 文件不存在：{filepath}")
        return [], 0, 0
    
    total = success + skipped
    print(f"\n📊 统计：成功 {success} 行，跳过 {skipped} 行，总计 {total} 行")
    return cities, success, skipped

# 测试
data, succ, skip = load_cities_robust("data/cities_climate.csv")
print(f"\n前 3 个城市：")
for c in data[:3]:
    print(" ", c)

In [ ]:
# ===== 课后作业 3 参考答案 =====
import math

def geographic_distance(lat1, lon1, lat2, lon2):
    """计算两点的地理距离（公里），带参数验证。
    
    参数：
        lat1, lon1: 第一个点的纬度、经度
        lat2, lon2: 第二个点的纬度、经度
    
    抛出：
        ValueError: 当纬度或经度超出有效范围时
    """
    # 参数验证
    for lat, name in [(lat1, "lat1"), (lat2, "lat2")]:
        if not (-90 <= lat <= 90):
            raise ValueError(f"纬度 {name} 必须在 -90 到 90 之间，传入 {lat}")
    
    for lon, name in [(lon1, "lon1"), (lon2, "lon2")]:
        if not (-180 <= lon <= 180):
            raise ValueError(f"经度 {name} 必须在 -180 到 180 之间，传入 {lon}")
    
    # 计算
    avg_lat_rad = math.radians((lat1 + lat2) / 2)
    dlat = lat1 - lat2
    dlon = (lon1 - lon2) * math.cos(avg_lat_rad)
    return 111 * math.sqrt(dlat ** 2 + dlon ** 2)

# 测试 5 个输入
tests = [
    (39.90, 116.40, 31.23, 121.47),     # 正常：北京-上海
    (90, 0, -90, 0),                     # 边界：极点
    (100, 0, 0, 0),                      # 错误：纬度超范围
    (0, 0, 0, 200),                      # 错误：经度超范围
    (45, 120, 30, 110),                  # 正常
]

for i, (la1, lo1, la2, lo2) in enumerate(tests, 1):
    print(f"\n--- 测试 {i}: ({la1}, {lo1}) → ({la2}, {lo2}) ---")
    try:
        d = geographic_distance(la1, lo1, la2, lo2)
        print(f"  ✅ 距离：{d:.1f} 公里")
    except ValueError as e:
        print(f"  ❌ 错误：{e}")